# 05 RoBERTa baseline (optional, runs locally)

Fine-tunes `roberta-base` on the round-2 human gold labels with 5-fold cross-validation (B5), and trains one on GPT-4o (C3) labels for the other pool reviews (B6). Needs internet access to Hugging Face for the first download. CPU is fine: about 20 to 40 minutes. No OpenAI cost.

After it finishes, re-run `analysis_round2.py` (last cell) so the RoBERTa rows appear in `results/04_human_annotation_round2/C_system_metrics.csv`.

In [ ]:
import sys, subprocess
try:
    import transformers, torch, sklearn
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', 'torch', 'scikit-learn'])
sys.path.insert(0, '../code')

In [ ]:
import pandas as pd, config as C, baselines_transformer as BT
pool = pd.read_csv(C.DATA_DIR / 'pool.csv')
gold = pd.read_csv(C.DATA_DIR / 'gold_labels_round2.csv')   # written by analysis_round2.py
print(len(gold), gold.forced_choice.value_counts().to_dict())

In [ ]:
b5 = BT.roberta_cv(gold, pool)          # B5: 5-fold CV on the human labels
b5.label.value_counts()

In [ ]:
ids = set(pd.read_csv(C.DATA_DIR / 'annotation_set_round2.csv').review_uid)
llm = pd.read_csv(C.RESULTS_DIR / '03_llm_outputs' / 'pool__C3_rag_full__run0.csv')
b6 = BT.roberta_silver(ids, pool, llm)   # B6: trained on GPT-4o labels, tested on the annotated reviews
b6.label.value_counts()

In [ ]:
import analysis_round2 as A
res, met = A.run()
A.make_figures(None, met, res)
met[met.subset == 'all agreed'][['name', 'accuracy', 'kappa', 'precision_regret', 'recall_regret', 'f1_regret']].round(3)